# Tandem repeat tracts: M0 against M1

How the repeat annotation changes between the two models, for every sample that
has both. Every table and plot compares them.

**Input:** `results/repetitive_dna/{species}/{sample}/{model}_summary.tsv`. One row
per distinct repeat type with a `count` column, so a whole genome fits in a table
small enough to plot locally. `species` and `sample` are columns in the file.

**Models:** M0 = mreps, exact repeats down to three copies, however short. 

M1 = TRF, which tolerates substitutions and indels between copies but only reports
tracts reaching its minimum alignment score, about 25 bp. 

In [ ]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)

results_dir      <- "../results/repetitive_dna"
models_compared  <- c("M0", "M1")  # add "M2" to bring the third model in
species_keep     <- NULL     # e.g. c("human", "chimp"); NULL keeps every species
max_length_shown <- 120      # x limit for the length distribution plots
motifs_shown     <- 6        # motifs per period in the per-motif plots, the most common first
length_breaks    <- c(0, 8, 12, 16, 24, 32, 48, 64, 128, Inf)   # tract length bins (bp)

options(repr.plot.width = 11, repr.plot.height = 6)

# One fixed colour per model, the same in every notebook in this folder.
model_colours <- c(M0 = "#2a78d6", M1 = "#eb6834", M2 = "#1baf7a")

# Quantiles of counted data, without expanding the counts into a vector.
weighted_quantile <- function(values, weights, probs) {
  o <- order(values)
  values <- values[o]
  cumulative <- cumsum(weights[o]) / sum(weights)
  vapply(probs, function(p) values[which(cumulative >= p)[1]], numeric(1))
}

# AG and CT are the same repeat read from opposite strands. Summaries written
# before canonicalise() in parse_trf.py collapsed the strands keep them apart;
# current ones hold only AG. These give both the one label "AG/CT" either way.
reverse_complement <- function(motif) {
  vapply(motif, function(m)
    paste(rev(chartr("ACGT", "TGCA", strsplit(m, "")[[1]])), collapse = ""),
    character(1), USE.NAMES = FALSE)
}

canonical_rotation <- function(motif) {
  vapply(motif, function(m) {
    bases <- strsplit(m, "")[[1]]
    min(vapply(seq_along(bases),
               function(i) paste(bases[c(i:length(bases), seq_len(i - 1))], collapse = ""),
               character(1)))
  }, character(1), USE.NAMES = FALSE)
}

strand_pair <- function(motif) {
  other <- canonical_rotation(reverse_complement(motif))
  ifelse(motif == other, motif,
         paste(pmin(motif, other), pmax(motif, other), sep = "/"))
}

# Labels for the tract length bins: "<8", "8-11", ..., ">=128".
bin_labels <- function(breaks) {
  low  <- head(breaks, -1)
  high <- tail(breaks, -1)
  ifelse(low == 0, paste0("<", high),
         ifelse(is.infinite(high), paste0(">=", low), paste0(low, "-", high - 1)))
}
length_bin <- function(length) cut(length, length_breaks, labels = bin_labels(length_breaks), right = FALSE)

## What is in the results folder

Every summary, and then the samples that have all the models compared here.

In [ ]:
summary_files <- list.files(results_dir, pattern = "_summary\\.tsv$",
                            recursive = TRUE, full.names = TRUE)
if (length(summary_files) == 0) stop("No summary files under ", results_dir)

inventory <- tibble(path = summary_files) |>
  mutate(
    parts   = strsplit(sub(paste0("^", results_dir, "/"), "", path), "/"),
    species = vapply(parts, `[`, character(1), 1),
    sample  = vapply(parts, `[`, character(1), 2),
    model   = sub("_summary\\.tsv$", "", basename(path)),
    present = "yes"
  ) |>
  select(species, sample, model, present) |>
  pivot_wider(names_from = model, values_from = present, values_fill = "-") |>
  arrange(species, sample)

cat(nrow(inventory), "samples,", n_distinct(inventory$species), "species\n")
print(inventory, n = 40)

paired <- inventory |>
  filter(if_all(all_of(models_compared), \(present) present == "yes")) |>
  select(species, sample)
if (!is.null(species_keep)) paired <- filter(paired, species %in% species_keep)
if (nrow(paired) == 0) stop("No sample has all of ", paste(models_compared, collapse = ", "))

cat("\n", nrow(paired), " samples have ", paste(models_compared, collapse = " and "), ":\n", sep = "")
print(paired, n = 40)

In [ ]:
# Both models for every paired sample. Reading the M1 tables takes a few
# seconds; they are much larger than the M0 ones.
tracts <- expand_grid(paired, model = models_compared) |>
  mutate(path = file.path(results_dir, species, sample, paste0(model, "_summary.tsv"))) |>
  pull(path) |>
  lapply(read_tsv, show_col_types = FALSE, progress = FALSE) |>
  bind_rows() |>
  mutate(length = round(k * copies),
         k      = factor(k),
         model  = factor(model, levels = models_compared),
         label  = paste(species, sample, sep = " / "))

# A motif and its reverse complement under one label, computed once per motif.
tracts <- tracts |>
  left_join(distinct(tracts, canonical_motif) |> mutate(motif = strand_pair(canonical_motif)),
            by = "canonical_motif")

cat(nrow(tracts), "rows,", format(sum(tracts$count), big.mark = ","), "tracts,",
    n_distinct(tracts$label), "samples\n")
head(tracts)

## How much each model finds

Tracts and the sequence they cover, per sample and period. `tract_ratio` and
`base_ratio` are the second model over the first: 0.01 means a hundredth as many
tracts.

In [ ]:
model_k <- tracts |>
  group_by(species, sample, label, model, k) |>
  summarise(
    tracts        = sum(count),
    bases         = sum(length * count),
    min_length    = min(length),
    median_length = weighted_quantile(length, count, 0.5),
    mean_length   = sum(length * count) / sum(count),
    q99_length    = weighted_quantile(length, count, 0.99),
    .groups = "drop"
  )

first  <- models_compared[1]
second <- models_compared[2]
model_k |>
  select(label, model, k, tracts, bases, min_length, median_length, mean_length) |>
  mutate(mean_length = round(mean_length, 1)) |>
  pivot_wider(names_from = model, values_from = c(tracts, bases, min_length, median_length, mean_length)) |>
  mutate(tract_ratio = round(.data[[paste0("tracts_", second)]] / .data[[paste0("tracts_", first)]], 3),
         base_ratio  = round(.data[[paste0("bases_", second)]] / .data[[paste0("bases_", first)]], 3)) |>
  print(n = 100, width = Inf)

model_k |>
  select(label, model, k, tracts, bases) |>
  pivot_longer(c(tracts, bases), names_to = "statistic", values_to = "value") |>
  mutate(statistic = recode(statistic, tracts = "tracts", bases = "bases covered")) |>
  ggplot(aes(x = k, y = value, colour = model)) +
  geom_point(position = position_jitterdodge(jitter.width = 0.15, jitter.height = 0, dodge.width = 0.6),
             alpha = 0.8, size = 1.8) +
  scale_colour_manual(values = model_colours) +
  scale_y_log10() +
  facet_wrap(~ statistic, scales = "free_y") +
  labs(title = "per period, one point per sample") +
  theme_minimal()

## Tract length

The length distribution of each model's tracts, one line per sample. Counts are
turned into a fraction per sample, model and period, so samples are comparable
whatever the assembly size.

In [ ]:
length_share <- tracts |>
  group_by(label, model, k, length) |>
  summarise(count = sum(count), .groups = "drop") |>
  group_by(label, model, k) |>
  mutate(fraction = count / sum(count)) |>
  ungroup()

# The whole range, so M1's long tracts are visible; the x axis is logarithmic
# because they reach the megabase satellite arrays. Each length is its own
# point, so the far tail is one or a few tracts and looks ragged; the next plot
# (tracts at least this long) is the smooth version of the same thing.
ggplot(length_share,
       aes(x = length, y = fraction, colour = model, group = interaction(label, model))) +
  geom_line(alpha = 0.6, linewidth = 0.4) +
  scale_colour_manual(values = model_colours) +
  scale_x_log10() +
  scale_y_log10() +
  facet_wrap(~ k, scales = "free_x",
             labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "Tract length distribution") +
  theme_minimal()

In [ ]:
# The same as absolute counts: tracts at least this long. Where the lines cross,
# the tolerant model starts reporting more tracts of that length or longer.
at_least <- length_share |>
  group_by(label, model, k) |>
  arrange(desc(length), .by_group = TRUE) |>
  mutate(tracts_at_least = cumsum(count)) |>
  ungroup()

ggplot(at_least, aes(x = length, y = tracts_at_least, colour = model,
                     group = interaction(label, model))) +
  geom_step(direction = "vh", alpha = 0.6, linewidth = 0.4) +
  scale_colour_manual(values = model_colours) +
  scale_x_log10() +
  scale_y_log10() +
  facet_wrap(~ k, labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "Tracts at least this long",
       x = "tract length (bp)", y = "tracts", colour = NULL) +
  theme_minimal()

# The shortest length from which the second model reports at least as many
# tracts as the first (NA: not within 500 bp).
grid <- seq_len(500)
tracts |>
  group_by(label, k) |>
  group_modify(function(d, key) {
    at_least_of <- function(model) {
      x <- filter(d, model == !!model)
      vapply(grid, function(L) sum(x$count[x$length >= L]), numeric(1))
    }
    n_first  <- at_least_of(first)
    n_second <- at_least_of(second)
    tibble(crossover_bp = grid[which(n_second >= n_first & n_second > 0)[1]])
  }) |>
  ungroup() |>
  pivot_wider(names_from = k, values_from = crossover_bp, names_prefix = "k = ") |>
  print(n = 40)

In [ ]:
# Median, mean and 99th percentile tract length per period, one point per
# sample. The mean sits above the median because the length distribution has a
# long tail, and it is the one a rate per tract base is built on.
model_k |>
  select(label, model, k, median_length, mean_length, q99_length) |>
  pivot_longer(c(median_length, mean_length, q99_length), names_to = "statistic", values_to = "bp") |>
  mutate(statistic = factor(recode(statistic,
                                   median_length = "median",
                                   mean_length = "mean",
                                   q99_length = "99th percentile"),
                            levels = c("median", "mean", "99th percentile"))) |>
  ggplot(aes(x = k, y = bp, colour = model)) +
  geom_point(position = position_jitterdodge(jitter.width = 0.15, jitter.height = 0, dodge.width = 0.6),
             alpha = 0.8, size = 1.8) +
  scale_colour_manual(values = model_colours) +
  scale_y_log10() +
  facet_wrap(~ statistic) +
  labs(title = "Tract length per period",
       x = "period k", y = "tract length (bp)", colour = NULL) +
  theme_minimal()

## Sequence covered by tracts

How much of the assembly each model calls repetitive, and where that sequence
sits by tract length.

In [ ]:
per_sample <- tracts |>
  group_by(species, sample, label, model) |>
  summarise(tracts = sum(count), bases = sum(length * count), .groups = "drop")

per_sample |>
  select(label, model, tracts, bases) |>
  pivot_wider(names_from = model, values_from = c(tracts, bases)) |>
  mutate(base_ratio = round(.data[[paste0("bases_", second)]] / .data[[paste0("bases_", first)]], 3)) |>
  arrange(desc(.data[[paste0("bases_", first)]])) |>
  print(n = 40, width = Inf)

options(repr.plot.width = 11, repr.plot.height = 7)
ggplot(per_sample, aes(x = reorder(label, bases, max), y = bases / 1e6, fill = model)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  scale_fill_manual(values = model_colours) +
  coord_flip() +
  labs(title = "Total sequence covered by tandem repeats",
       x = NULL, y = "Mb", fill = NULL) +
  theme_minimal()

# The same split by period.
ggplot(model_k, aes(x = k, y = bases / 1e6, fill = model)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  scale_fill_manual(values = model_colours) +
  facet_wrap(~ label, scales = "free_y") +
  labs(title = "Sequence covered per period", x = "period k", y = "Mb", fill = NULL) +
  theme_minimal()
options(repr.plot.width = 11, repr.plot.height = 6)

In [ ]:
# Where each model's repeat sequence sits by tract length, as a share of that
# model's sequence at that period. One line per sample, length continuous: the
# x axis is logarithmic because M1's tracts run from 25 bp to the megabase
# satellite arrays.
bases_by_length <- tracts |>
  group_by(label, model, k, length) |>
  summarise(bases = sum(length * count), .groups = "drop") |>
  group_by(label, model, k) |>
  mutate(share = bases / sum(bases)) |>
  ungroup()

ggplot(bases_by_length, aes(x = length, y = share, colour = model,
                            group = interaction(label, model))) +
  geom_line(alpha = 0.6, linewidth = 0.4) +
  scale_colour_manual(values = model_colours) +
  scale_x_log10() +
  scale_y_log10() +
  facet_wrap(~ k, labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "Repeat sequence by tract length",
       subtitle = "share of the model's repeat sequence at that period",
       x = "tract length (bp)", y = "share of bases", colour = NULL) +
  theme_minimal()

## Motifs and their tract lengths, per period

For each period, the most common motifs and how long their tracts are in each
model. Samples are pooled here, so a motif's line is the whole paired set; the
plots above show how much samples vary. A motif and its reverse complement share
one label (AC/GT), since they are the same repeat read from opposite strands.

In [ ]:
# The motifs carrying the most repeat sequence in each period, over both models.
top_motifs <- tracts |>
  group_by(k, motif) |>
  summarise(bases = sum(length * count), .groups = "drop_last") |>
  slice_max(bases, n = motifs_shown) |>
  ungroup() |>
  select(k, motif)

motif_length <- tracts |>
  inner_join(top_motifs, by = c("k", "motif")) |>
  group_by(k, motif, model) |>
  summarise(
    tracts = sum(count),
    bases  = sum(length * count),
    q25    = weighted_quantile(length, count, 0.25),
    median = weighted_quantile(length, count, 0.5),
    q75    = weighted_quantile(length, count, 0.75),
    q99    = weighted_quantile(length, count, 0.99),
    .groups = "drop"
  )

print(arrange(motif_length, k, motif, model), n = 100)

options(repr.plot.width = 12, repr.plot.height = 6)
ggplot(motif_length, aes(x = reorder(motif, -bases), colour = model)) +
  geom_linerange(aes(ymin = q25, ymax = q75), position = position_dodge(width = 0.6), linewidth = 0.8) +
  geom_point(aes(y = median), position = position_dodge(width = 0.6), size = 2) +
  geom_point(aes(y = q99), position = position_dodge(width = 0.6), size = 1.4, shape = 1) +
  scale_colour_manual(values = model_colours) +
  scale_y_log10() +
  facet_grid(. ~ k, scales = "free_x", space = "free_x",
             labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "Tract length per motif",
       subtitle = "point: median, bar: 25th-75th percentile, open point: 99th percentile; motifs ordered by sequence covered",
       x = "motif (both strands)", y = "tract length (bp)", colour = NULL) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
options(repr.plot.width = 11, repr.plot.height = 6)

In [ ]:
# The full length distribution behind those summaries: one panel per motif, one
# plot per period.
motif_share <- tracts |>
  inner_join(top_motifs, by = c("k", "motif")) |>
  group_by(k, motif, model, length) |>
  summarise(count = sum(count), .groups = "drop") |>
  group_by(k, motif, model) |>
  mutate(fraction = count / sum(count)) |>
  ungroup()

options(repr.plot.width = 12, repr.plot.height = 6)
for (period in levels(droplevels(motif_share$k))) {
  print(motif_share |>
    filter(k == period, length <= max_length_shown) |>
    ggplot(aes(x = length, y = fraction, colour = model)) +
    geom_line(linewidth = 0.5) +
    scale_colour_manual(values = model_colours) +
    scale_y_log10() +
    facet_wrap(~ motif, scales = "free_y") +
    labs(title = paste0("Tract length distribution per motif, k = ", period),
         subtitle = paste("fraction of that motif's tracts at each length, samples pooled; x cut at",
                          max_length_shown, "bp"),
         x = "tract length (bp)", y = "fraction of tracts", colour = NULL) +
    theme_minimal())
}
options(repr.plot.width = 11, repr.plot.height = 6)

In [ ]:
# Which motifs each model's sequence sits in: the share of the period's repeat
# sequence per motif, samples pooled. Motifs above the dashed line make up more
# of the second model's sequence than of the first's.
motif_bases <- tracts |>
  group_by(model, k, motif) |>
  summarise(bases = sum(length * count), .groups = "drop") |>
  group_by(model, k) |>
  mutate(share = bases / sum(bases)) |>
  ungroup() |>
  select(-bases) |>
  pivot_wider(names_from = model, values_from = share, values_fill = 0)

both <- filter(motif_bases, .data[[first]] > 0, .data[[second]] > 0)

ggplot(both, aes(x = .data[[first]], y = .data[[second]])) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey60") +
  geom_point(colour = "#52514e", alpha = 0.7, size = 1.8) +
  geom_text(data = filter(both, pmax(.data[[first]], .data[[second]]) >= 0.02),
            aes(label = motif), size = 3, vjust = -0.8, check_overlap = TRUE) +
  scale_x_log10(expand = expansion(mult = c(0.05, 0.2))) +
  scale_y_log10(expand = expansion(mult = c(0.05, 0.1))) +
  facet_wrap(~ k, labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = paste("Share of the period's repeat sequence per motif,", first, "against", second),
       subtitle = "samples pooled; labelled: at least 2% in either model",
       x = paste("share in", first), y = paste("share in", second)) +
  theme_minimal()

# Motifs found by one model only, with their share of that model's sequence.
motif_bases |>
  filter(.data[[first]] == 0 | .data[[second]] == 0) |>
  mutate(only_in = ifelse(.data[[first]] == 0, second, first),
         share   = pmax(.data[[first]], .data[[second]])) |>
  group_by(k, only_in) |>
  summarise(motifs = n(), share_of_sequence = round(sum(share), 4),
            largest = paste(head(motif[order(-share)], 3), collapse = ", "),
            .groups = "drop") |>
  print()

## Why the tolerant model's tracts are longer

M0 tracts are 100% pure by definition, so this is the one thing the models cannot
be compared on directly. `model_purity` compares every base of a tract with the
motif tiled from its start, so an indel inside the tract lowers it for everything
after; `trf_purity` is TRF's own alignment score, which forgives indels. Long
tracts that are rarely pure are the exact runs M0 reports separately.

In [ ]:
impure <- tracts |>
  filter(model != "M0") |>
  mutate(length_bin = length_bin(length))

impure |>
  group_by(label, model, k) |>
  summarise(
    tracts              = sum(count),
    pure_tract_share    = round(sum(count[model_purity == 100]) / sum(count), 3),
    pure_base_share     = round(sum((length * count)[model_purity == 100]) / sum(length * count), 3),
    median_model_purity = weighted_quantile(model_purity, count, 0.5),
    median_trf_purity   = weighted_quantile(trf_purity, count, 0.5),
    .groups = "drop"
  ) |>
  print(n = 100, width = Inf)

impure |>
  group_by(label, model, k, length_bin) |>
  summarise(
    `share of tracts fully pure` = sum(count[model_purity == 100]) / sum(count),
    `median model_purity (%)`    = weighted_quantile(model_purity, count, 0.5),
    .groups = "drop"
  ) |>
  pivot_longer(c(`share of tracts fully pure`, `median model_purity (%)`),
               names_to = "statistic", values_to = "value") |>
  ggplot(aes(x = length_bin, y = value, colour = model, group = interaction(label, model))) +
  geom_line(alpha = 0.6, linewidth = 0.4) +
  geom_point(size = 0.8, alpha = 0.6) +
  scale_colour_manual(values = model_colours) +
  facet_grid(statistic ~ k, scales = "free_y",
             labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "Purity of the tolerant model's tracts by tract length, one line per sample",
       x = "tract length (bp)", y = NULL, colour = NULL) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))